# Chapter 1. Foundations of cheminformatics - Part 3

## 1.3. RDKit

### 1.3.1. Introduction

**Learning objectives**

- Parse and draw molecular graphs with RDKit, and validate a structure against its identity.
- Read SMILES atoms, bonds, branches, rings, charges, isotopes, and stereochemistry.
- Distinguish molecular graphs, 2D depictions, and 3D conformers.
- Use SMARTS for a substructure query and distinguish reaction records from transformation templates.
- Explain what XYZ, MOL, SDF, PDBx/mmCIF, and CIF files preserve or omit.

**Prerequisites:** Parts 1 and 2. The chemical vocabulary needed here is introduced below; no previous cheminformatics is assumed. Run cells in order; all required data are embedded below. The interactive 3D viewer is an optional extension.

RDKit is an open-source toolkit with a C++ core and Python interface. Here we use it to manipulate **molecular graphs**: atoms are nodes and bonds are edges. Coordinates, properties, and metadata can be attached to that graph. A successful parse means RDKit accepted the representation; it does not prove that the input has the intended chemical identity or is experimentally stable.

### Start with the chemical picture

A molecule can be represented as a **graph**: an atom is a labeled point and a bond connects two points. Hydrogens are often omitted from drawings but still contribute to composition. A **molecular formula** counts elements; **connectivity** states which atoms are joined; **stereochemistry** states arrangements that connectivity alone does not distinguish.

| Word | First picture | Why it matters |
| --- | --- | --- |
| Isomers | Same formula, different structures | A mass or formula match alone can identify the wrong compound |
| Protonation state | A hydrogen ion has been added or removed | Charge and composition can change with chemical conditions |
| Conformer | Same molecular identity, a different spatial arrangement | One SMILES can lead to several 3D shapes |
| Descriptor | A number calculated from a representation | Its interpretation depends on what the representation includes |

**Core route:** graph versus coordinates → a small SMILES syntax vocabulary → stereochemistry and canonicalization → the identity audit and substructure search. The later paclitaxel example applies the identity checks to a larger real compound; its long SMILES is not a string to memorize. Detailed reaction templates, crystallographic formats, and the long XYZ mapping helper are **deeper reference**; read their conclusions first and return to the implementation later.

**Predict:** can ethanol (`CCO`) and dimethyl ether (`COC`) have the same formula but different bonds? Keep this question for the identity audit.

### 1.3.2. Installation

Use the course environment in the README. For a separate installation, the supported PyPI package is `rdkit`; `rdkit-pypi` is the old package name. Run setup in a terminal, before opening the notebook. [RDKit installation](https://www.rdkit.org/docs/Install.html).

```text
python -m pip install rdkit
```

For more information about RDKit, see [documentation](https://www.rdkit.org/docs/)

### 1.3.3. Import RDKit

To use RDKit in your Python code, you need to import the library:

In [ ]:
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors
from rdkit.Chem import rdCIPLabeler, rdChemReactions, rdDetermineBonds
from rdkit.Chem import rdDepictor

print("RDKit version:", rdBase.rdkitVersion)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def molecule_from_smiles(smiles):
    """Parse a single non-empty SMILES, or raise a useful error."""
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("SMILES must be a non-empty string.")
    parameters = Chem.SmilesParserParams()
    parameters.parseName = False
    parameters.allowCXSMILES = False
    mol = Chem.MolFromSmiles(smiles, parameters)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles!r}")
    if mol.GetNumAtoms() == 0:
        raise ValueError("The representation contains no atoms.")
    return mol

## 1.4. Simplified Molecular Input Line Entry System (SMILES)

### 1.4.1. Introduction

SMILES (Simplified Molecular Input Line Entry System) encodes connectivity and can include isotopes and stereochemical information. Ordinary SMILES does **not** contain coordinates. For example, `O=C(C)Oc1ccccc1C(=O)O` describes aspirin.

The following examples demonstrate the syntax with RDKit. We use a checked parser because `Chem.MolFromSmiles` normally returns `None` for invalid input. Its default sanitization checks include valence and aromaticity. It should not be disabled merely to make an incorrect structure load. [RDKit getting started](https://www.rdkit.org/docs/GettingStartedInPython.html).

**Three distinct objects**

1. The molecular **graph** records atoms, bonds, and chemical annotations.
2. A **2D depiction** places that graph on a page. Drawing software chooses coordinates for readability.
3. A **3D conformer** assigns one spatial arrangement to the atoms. A molecule may have many conformers.

Neither a clean drawing nor a rotatable viewer establishes that coordinates represent an experimental or lowest-energy structure.

**Generate real 3D coordinates**

Use a small molecule, ethanol, to make the steps easy to inspect. Add explicit hydrogens, generate a conformer with ETKDGv3, then minimize with MMFF. Check both return codes. A fixed seed makes a run reproducible within an environment; toolkit versions can still change coordinates. This is one model conformer, not a conformational search. [RDKit distance geometry](https://www.rdkit.org/docs/source/rdkit.Chem.rdDistGeom.html).

In [ ]:
ethanol_3d = Chem.AddHs(molecule_from_smiles("CCO"))
embedding_parameters = AllChem.ETKDGv3()
embedding_parameters.randomSeed = 2026
embedding_parameters.numThreads = 1
conformer_id = AllChem.EmbedMolecule(ethanol_3d, embedding_parameters)
if conformer_id < 0:
    raise RuntimeError("3D coordinate generation failed.")
if not AllChem.MMFFHasAllMoleculeParams(ethanol_3d):
    raise ValueError("MMFF parameters are unavailable for this molecule.")
optimization_status = AllChem.MMFFOptimizeMolecule(ethanol_3d, maxIters=1000)
if optimization_status != 0:
    raise RuntimeError(f"MMFF did not converge; status={optimization_status}")
assert ethanol_3d.GetConformer().Is3D()
print("3D conformer generated and MMFF minimization converged.")

In [ ]:
# A static 3D view works offline and is saved in notebook outputs.
coordinates = ethanol_3d.GetConformer().GetPositions()
fig = plt.figure(figsize=(7, 5.5))
ax = fig.add_subplot(111, projection="3d")
colors = {"C": "#444444", "O": "#d62728", "H": "#c9c9c9"}
for symbol in ("C", "O", "H"):
    indices = [atom.GetIdx() for atom in ethanol_3d.GetAtoms() if atom.GetSymbol() == symbol]
    points = coordinates[indices]
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors[symbol], s=80, label=symbol)
for bond in ethanol_3d.GetBonds():
    points = coordinates[[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
    ax.plot(points[:, 0], points[:, 1], points[:, 2], color="gray")
# Equal coordinate ranges avoid visually stretching bond lengths.
center = coordinates.mean(axis=0)
half_width = np.ptp(coordinates, axis=0).max() / 2 + 0.4
ax.set(xlim=(center[0]-half_width, center[0]+half_width),
       ylim=(center[1]-half_width, center[1]+half_width),
       zlim=(center[2]-half_width, center[2]+half_width),
       xlabel="x (angstrom)", ylabel="y (angstrom)", zlabel="z (angstrom)",
       title="One calculated ethanol conformer")
ax.set_box_aspect((1, 1, 1), zoom=0.8)
ax.legend()
plt.show()

**Optional interactive 3D viewer**

The static figure above works offline. `SHOW_INTERACTIVE = False` keeps this extension disabled; set it to `True` only when browser JavaScript and the 3Dmol.js CDN are available. A viewer displays supplied coordinates; it does not generate a 3D geometry. [py3Dmol documentation](https://github.com/3dmol/3Dmol.js/tree/master/py3Dmol).

In [ ]:
SHOW_INTERACTIVE = False
if SHOW_INTERACTIVE:
    import py3Dmol
    view = py3Dmol.view(width=640, height=400)
    view.addModel(Chem.MolToMolBlock(ethanol_3d), "mol")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.show()
else:
    print("Interactive viewer disabled; the static conformer plot above is the offline view.")

### 1.4.2. Atom

**Atoms and brackets**

Write element symbols with correct case: `Cl`, not `CL`. Many neutral atoms in the organic subset (`B C N O P S F Cl Br I`) can omit brackets when normal valence rules apply. Brackets specify other elements, isotopes, charges, or explicit hydrogen counts.

`[Na]` is a neutral sodium atom; `[Na+]` is the sodium ion. Neither isolated symbol describes a bulk metallic lattice. [Daylight SMILES rules](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html#RTFToC4).

In [ ]:
gold_atom = molecule_from_smiles("[Au]")
Draw.MolToImage(gold_atom, size=(150, 100))

In [ ]:
sodium_ion = molecule_from_smiles("[Na+]")
Draw.MolToImage(sodium_ion, size=(150, 100))

In [ ]:
# The prefix is a mass number, not a mass in Da: carbon-13 methane.
carbon13_methane = molecule_from_smiles("[13CH4]")
print("Isotope mass number:", carbon13_methane.GetAtomWithIdx(0).GetIsotope())
Draw.MolToImage(carbon13_methane, size=(150, 100))

**Hydrogens**

For an unbracketed atom, RDKit infers attached hydrogens from valence and bonds. Thus `C`, `N`, and `O` below describe methane, ammonia, and water. Bracket hydrogens must be specified when required: `[NH4+]` is ammonium. `Chem.AddHs` adds hydrogen atoms as explicit nodes in the graph.

In [ ]:
# Methane (CH4)
CH4 = molecule_from_smiles('C')
Draw.MolToImage(CH4, size=(100, 100))

In [ ]:
# Ammonia (NH3)
NH3 = molecule_from_smiles('N')
Draw.MolToImage(NH3, size=(100, 100))

In [ ]:
# Water (H2O)
water = molecule_from_smiles('O')
Draw.MolToImage(water, size=(150, 100))

**Aromatic atoms**

Lowercase atom symbols encode aromatic atoms, such as `c` and `n`. Uppercase symbols can also describe an aromatic molecule in Kekule form: RDKit recognizes `C1=CC=CC=C1` as benzene. Aromaticity is a toolkit perception model; do not infer that every uppercase representation is nonaromatic.

In [ ]:
# Benzene
benzene = molecule_from_smiles('c1ccccc1')
Draw.MolToImage(benzene, size=(100, 100))

In [ ]:
# Hexane
hexane = molecule_from_smiles('CCCCCC')
Draw.MolToImage(hexane, size=(150, 150))

In [ ]:
# Cyclohexane
cyclohexane = molecule_from_smiles('C1CCCCC1')
Draw.MolToImage(cyclohexane, size=(100, 100))

**Formal charges**

Inside brackets, `+` and `-` indicate formal charge; omit the magnitude for one unit. Examples: `[OH-]`, `[OH3+]`, `[O-2]`, and `[Au+3]`. Formal charge is an integer bookkeeping assignment, distinct from a model-dependent partial charge.

In [ ]:
# OH-
hydroxide_ion = molecule_from_smiles('[OH-]')
Draw.MolToImage(hydroxide_ion, size=(150, 100))

In [ ]:
# Hydronium ion
hydronium_ion = molecule_from_smiles('[OH3+]')
Draw.MolToImage(hydronium_ion, size=(100, 100))

In [ ]:
# Oxide ion
oxide_ion = molecule_from_smiles('[O-2]')
Draw.MolToImage(oxide_ion, size=(100, 100))

In [ ]:
# Gold ion
gold_ion = molecule_from_smiles('[Au+3]')
Draw.MolToImage(gold_ion, size=(100, 100))

### 1.4.3. Bond

| Bond type | SMILES symbol |
|---|---|
| Single | `-` |
| Double | `=` |
| Triple | `#` |
| Aromatic | `:` |

An omitted bond between adjacent atoms is normally single, or aromatic between aromatic atoms. The bond order belongs to the graph; a drawn line's length is not a bond-length measurement.

In [ ]:
# Hydrogen molecule
H2 = molecule_from_smiles('[H][H]')
Draw.MolToImage(H2, size=(100, 100))

In [ ]:
# Single bonds are implied
ethanol = molecule_from_smiles('CCO')
Draw.MolToImage(ethanol, size=(150, 150))

In [ ]:
# Double bond
one_butene = molecule_from_smiles('C=CCC')
Draw.MolToImage(one_butene, size=(150, 150))

In [ ]:
# Acetylene
acetylene = molecule_from_smiles('C#C')
Draw.MolToImage(acetylene, size=(100, 50))

In [ ]:
# Aniline; colons explicitly indicate aromatic bonds.
aniline = molecule_from_smiles("Nc1:c:c:c:c:c:1")
Draw.MolToImage(aniline, size=(200, 150))

### 1.4.4. Branch

A branch in parentheses starts from the atom immediately before the opening parenthesis. Return to that atom when the branch closes. Trace `CC(C)CC`: the parent chain has four carbons and a methyl branch at carbon 2.

In [ ]:
# 2-Methylpropane
two_methylpropane = molecule_from_smiles('CC(C)C')
Draw.MolToImage(two_methylpropane, size=(150, 100))

In [ ]:
# 2-Methylbutane has FIVE carbons; CC(C)C(C)C would be 2,3-dimethylbutane.
two_methylbutane = molecule_from_smiles("CC(C)CC")
assert rdMolDescriptors.CalcMolFormula(two_methylbutane) == "C5H12"
Draw.MolToImage(two_methylbutane, size=(200, 150))

In [ ]:
# Acetone
acetone = molecule_from_smiles('CC(=O)C')
Draw.MolToImage(acetone, size=(150, 100))

### 1.4.5. Cyclic Structures

Paired ring labels add a bond between the labeled atoms. Digits are closure labels, not atom counts or atom indices: `C1CCCCC1` and `C2CCCCC2` both encode cyclohexane. More than one closure can attach to an atom; `%10` is an example of a two-digit label.

In [ ]:
# Cyclic-aliphatic
aliphatic  = molecule_from_smiles('C1CCCCC1')
Draw.MolToImage(aliphatic, size=(100, 100))

In [ ]:
# Pyrrole: [nH] supplies the hydrogen on its aromatic nitrogen.
pyrrole = molecule_from_smiles("c1cc[nH]c1")
assert rdMolDescriptors.CalcMolFormula(pyrrole) == "C4H5N"
Draw.MolToImage(pyrrole, size=(180, 150))

In [ ]:
# Bicyclic compounds
bicyclic = molecule_from_smiles('C1CC2CCC2C1')
Draw.MolToImage(bicyclic, size=(100, 100))

### 1.4.6. Disconnected Structures

A dot separates disconnected components: no bond is implied across `.`. Salts are commonly represented as charged components, such as `[Na+].[Cl-]`. This graph does not describe crystal packing or solution ion pairing.

In [ ]:
sodium_chloride = molecule_from_smiles("[Na+].[Cl-]")
assert len(Chem.GetMolFrags(sodium_chloride)) == 2
assert Chem.GetFormalCharge(sodium_chloride) == 0
Draw.MolToImage(sodium_chloride, size=(180, 100))

In [ ]:
# Sodium phenoxide
sodium_phenoxide = molecule_from_smiles('[Na+].[O-]c1ccccc1')
Draw.MolToImage(sodium_phenoxide, size=(150, 150))

### 1.4.7. Double-bond configuration

Directional bonds `/` and `\` encode relative stereochemistry across a double bond. For the simple sequence `C/C=C/C`, the methyl groups are opposite: trans (E)-but-2-ene. `C/C=C\C` is cis (Z)-but-2-ene.

These cis/Z and trans/E pairings hold for this example. In general, **E/Z** uses Cahn-Ingold-Prelog (CIP) priorities, and **cis/trans** compares specified substituents; do not equate them universally. Missing slash symbols leave the configuration unspecified, not necessarily a mixture. Use a Python raw string (`r"..."`) when a SMILES contains backslashes.

In [ ]:
cis_2_butene = molecule_from_smiles(r"C/C=C\C")
Draw.MolToImage(cis_2_butene, size=(200, 150))

In [ ]:
trans_2_butene = molecule_from_smiles("C/C=C/C")
Draw.MolToImage(trans_2_butene, size=(200, 150))

In [ ]:
for label, mol in [("cis-but-2-ene", cis_2_butene), ("trans-but-2-ene", trans_2_butene)]:
    double_bond = next(bond for bond in mol.GetBonds() if bond.GetBondType() == Chem.BondType.DOUBLE)
    print(label, double_bond.GetStereo())
assert next(b for b in cis_2_butene.GetBonds() if b.GetBondType() == Chem.BondType.DOUBLE).GetStereo() == Chem.BondStereo.STEREOZ
assert next(b for b in trans_2_butene.GetBonds() if b.GetBondType() == Chem.BondType.DOUBLE).GetStereo() == Chem.BondStereo.STEREOE

### 1.4.8. Chirality

For tetrahedral stereochemistry, `@` and `@@` encode opposite orientations **relative to the neighbor order in the SMILES**, not fixed R/S labels. Reordering neighbors can require changing `@` to `@@` while preserving the same stereoisomer. The viewpoint and implicit hydrogen order are specified in the [SMILES stereochemistry rules](https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html#RTFToC12).

Assign CIP labels from the parsed graph rather than guessing from the symbol. We use RDKit's accurate CIP labeler for the simple examples below. An unspecified stereocenter has no assigned R/S identity. [RDKit CIP labeler](https://www.rdkit.org/docs/source/rdkit.Chem.rdCIPLabeler.html).

In [ ]:
def cip_labels(mol):
    labeled = Chem.Mol(mol)
    rdCIPLabeler.AssignCIPLabels(labeled)
    return [(atom.GetIdx(), atom.GetProp("_CIPCode"))
            for atom in labeled.GetAtoms() if atom.HasProp("_CIPCode")]

R_alanine = molecule_from_smiles("C[C@H](C(=O)O)N")
assert cip_labels(R_alanine) == [(1, "R")]
print("Atom index, CIP label:", cip_labels(R_alanine))
Draw.MolToImage(R_alanine, size=(220, 170))

In [ ]:
S_alanine = molecule_from_smiles("C[C@@H](C(=O)O)N")
assert cip_labels(S_alanine) == [(1, "S")]
print("Atom index, CIP label:", cip_labels(S_alanine))
Draw.MolToImage(S_alanine, size=(220, 170))

In [ ]:
# Here @@ gives R with THIS atom order.
R_2_chlorobutane = molecule_from_smiles("C[C@@H](Cl)CC")
assert cip_labels(R_2_chlorobutane) == [(1, "R")]
Draw.MolToImage(R_2_chlorobutane, size=(220, 170))

In [ ]:
S_2_chlorobutane = molecule_from_smiles("C[C@H](Cl)CC")
assert cip_labels(S_2_chlorobutane) == [(1, "S")]
Draw.MolToImage(S_2_chlorobutane, size=(220, 170))

Exchange the nitrogen and carboxyl branches in the R-alanine SMILES. The next example changes `@` to `@@` to retain R configuration:

In [ ]:
same_R_alanine = molecule_from_smiles("C[C@@H](N)C(=O)O")
assert cip_labels(same_R_alanine) == [(1, "R")]
assert Chem.MolToSmiles(same_R_alanine) == Chem.MolToSmiles(R_alanine)
print("Different traversal, same stereoisomer.")

### 1.4.9. Canonical SMILES

Many strings can describe the same graph. Canonicalization chooses one serialization using a particular toolkit's algorithm and settings; it is useful for comparing representations within a defined workflow. It is not a universal naming standard and can change between toolkit versions.

Canonicalization does **not** automatically neutralize charges, remove counterions, choose a tautomer, or supply missing stereochemistry. Keep isomeric output enabled when identity depends on stereochemistry or isotopes. [RDKit canonicalization and changes](https://www.rdkit.org/docs/BackwardsIncompatibleChanges.html).

In [ ]:
ethanol_strings = ["[CH3][CH2][OH]", "OCC", "CCO", "C(O)C", "C-C-O"]
ethanol_molecules = [molecule_from_smiles(text) for text in ethanol_strings]
canonical_forms = {Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
                   for mol in ethanol_molecules}
assert canonical_forms == {"CCO"}
Draw.MolsToGridImage(ethanol_molecules, legends=ethanol_strings, molsPerRow=3)

`Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)` produces canonical isomeric SMILES (these options are enabled by default). This serializes specified stereochemistry; it cannot recover stereochemistry absent from the input.

In [ ]:
mol = molecule_from_smiles("C(O)C")
canonical_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
print(canonical_smiles)

In [ ]:
r_text = Chem.MolToSmiles(R_alanine, isomericSmiles=True)
s_text = Chem.MolToSmiles(S_alanine, isomericSmiles=True)
print("R-alanine:", r_text)
print("S-alanine:", s_text)
assert r_text != s_text
# Suppressing stereo information makes these enantiomers indistinguishable.
assert Chem.MolToSmiles(R_alanine, isomericSmiles=False) == Chem.MolToSmiles(S_alanine, isomericSmiles=False)

### Research application: what counts as a duplicate structure?

A researcher merges two compound inventories before joining measurement tables. Comparing text alone misses alternative SMILES; comparing formulas alone merges isomers. Here the records are **deliberately chosen representations of real molecules**, not invented experimental measurements. We calculate the audit outcomes from RDKit.

We choose a conservative rule: a duplicate must have the same **canonical isomeric SMILES under this RDKit version**, preserving charges, isotopes, and specified stereochemistry. This does not reconcile tautomers, salts, or missing stereo. Retain original records and document any later standardization policy.

**Predict:** which pair should merge? Which pairs need the original source or assay conditions before any broader reconciliation?

In [ ]:
import pandas as pd

identity_pairs = [
    ("same ethanol", "CCO", "OCC"),
    ("ethanol / ether", "CCO", "COC"),
    ("acid / conjugate base", "CC(=O)O", "CC(=O)[O-]"),
    ("opposite alanine stereo", "C[C@H](N)C(=O)O", "C[C@@H](N)C(=O)O"),
    ("specified / missing stereo", "C[C@H](N)C(=O)O", "CC(N)C(=O)O"),
]
identity_rows = []
for label, left_text, right_text in identity_pairs:
    left_mol, right_mol = map(molecule_from_smiles, (left_text, right_text))
    identity_rows.append({"pair": label, "left_SMILES": left_text, "right_SMILES": right_text,
        "same text": left_text == right_text,
        "same formula": rdMolDescriptors.CalcMolFormula(left_mol) == rdMolDescriptors.CalcMolFormula(right_mol),
        "same canonical stereo graph": Chem.MolToSmiles(left_mol, isomericSmiles=True)
                                      == Chem.MolToSmiles(right_mol, isomericSmiles=True)})
identity_audit = pd.DataFrame(identity_rows)
assert identity_audit["same canonical stereo graph"].tolist() == [True, False, False, False, False]
identity_audit

In [ ]:
comparison_columns = ["same text", "same formula", "same canonical stereo graph"]
comparison = identity_audit[comparison_columns].to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.imshow(comparison, cmap="Blues", vmin=0, vmax=1, aspect="auto")
for row in range(len(identity_pairs)):
    for column in range(3):
        ax.text(column, row, "same" if comparison[row, column] else "different",
                ha="center", va="center", color="white" if comparison[row, column] else "#334155")
ax.set(xticks=range(3), xticklabels=["Raw text", "Element counts", "Canonical stereo graph"],
       yticks=range(len(identity_pairs)), yticklabels=identity_audit["pair"],
       title="Calculated identity audit: the comparison rule changes the answer")
ax.tick_params(length=0)
plt.show()

**Decision:** merge the alternative ethanol representations under this stated rule, while retaining their source identifiers. Do not merge the other pairs automatically. In particular, an unspecified stereocenter is incomplete information, not evidence for either a pure enantiomer or a racemic mixture.

**Explain:** the formula-only rule would confuse ethanol with ether and the two alanine enantiomers. A join using only raw strings would miss the ethanol duplicate. Correct software cannot recover identity information absent from a source record.

#### Deeper application: validate a named, complex compound

Now apply the same identity principles to paclitaxel. Read the two validation assertions and the resulting formula/mass outputs; do not try to memorize the long SMILES. An InChIKey is a compact identifier derived from an InChI chemical representation. We compare it with a trusted reference record as an additional identity check.

**Worked example: paclitaxel (Taxol)**

The SMILES below is a local copy from [PubChem CID 36314](https://pubchem.ncbi.nlm.nih.gov/compound/36314), checked on 2026-09-15. PubChem reports formula **C47H51NO14** and InChIKey **RCINICONZNJXQF-MZXODVADSA-N**. We check both: a formula alone cannot distinguish constitutional isomers or stereoisomers.

Before analyzing a named compound, compare its parsed graph to a trusted record; the caption alone is not an identity check.

In [ ]:
# PubChem CID 36314, isomeric SMILES; kept locally for reproducibility.
paclitaxel_smiles = (
    "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]"
    "([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)"
    "NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C"
)
taxol_mol = molecule_from_smiles(paclitaxel_smiles)
assert rdMolDescriptors.CalcMolFormula(taxol_mol) == "C47H51NO14"
assert Chem.MolToInchiKey(taxol_mol) == "RCINICONZNJXQF-MZXODVADSA-N"
print("Identity checks passed.")

In [ ]:
taxol_2d = Chem.Mol(taxol_mol)  # Keep the source object separate.
rdDepictor.Compute2DCoords(taxol_2d)
assert not taxol_2d.GetConformer().Is3D()
Draw.MolToImage(taxol_2d, size=(700, 500))

In [ ]:
molar_mass = Descriptors.MolWt(taxol_mol)
monoisotopic_mass = Descriptors.ExactMolWt(taxol_mol)
print("Formula:", rdMolDescriptors.CalcMolFormula(taxol_mol))
print(f"Average molar mass: {molar_mass:.2f} g/mol")
print(f"Monoisotopic molecular mass: {monoisotopic_mass:.5f} Da")

`MolWt` uses average atomic weights; `ExactMolWt` uses isotope-specific masses (the most abundant isotope when none is specified). Their numerical values answer different questions. Neither value alone is an observed mass-spectrometry peak; charge state and adduct composition also matter.

## 1.5. SMARTS queries and reaction representations

SMARTS describes a **query**, which asks whether a molecular graph contains a pattern. It extends SMILES-like syntax with constraints and logical operators, but the interpretation is not identical to a molecule SMILES. For example, SMARTS `[#6]` matches any carbon atom, `C` matches aliphatic carbon, and `c` matches aromatic carbon.

Create queries with `Chem.MolFromSmarts`, then use `HasSubstructMatch` or `GetSubstructMatches`. This is useful for functional-group searches as well as reaction templates. [Daylight SMARTS specification](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html).

In [ ]:
# X4 = four total neighbors (including H); X2 = two; H1 = one attached H.
alcohol_query = Chem.MolFromSmarts("[CX4][OX2H1]")
if alcohol_query is None:
    raise ValueError("Invalid alcohol SMARTS query.")
examples = {"ethanol": "CCO", "dimethyl ether": "COC", "acetic acid": "CC(=O)O",
            "phenol": "Oc1ccccc1", "ethylene glycol": "OCCO", "acetate": "CC(=O)[O-]"}
hits = {name: molecule_from_smiles(text).HasSubstructMatch(alcohol_query)
        for name, text in examples.items()}
print(hits)
assert hits == {"ethanol": True, "dimethyl ether": False, "acetic acid": False,
                "phenol": False, "ethylene glycol": True, "acetate": False}

### A visual catalog screen: “contains oxygen” versus “contains an alcohol”

Suppose the research question needs an O–H group attached to a saturated carbon. Counting oxygen atoms would also retrieve ethers, acids, and phenols. The SMARTS above encodes our stated local pattern; it is a query definition, not a prediction of reaction yield or compatibility.

The highlighted atoms below are the **matched C–O pairs**, including both pairs in ethylene glycol. A blank highlight means the chosen query did not match, even if the structure contains oxygen.

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import Image, display

query_molecules = [molecule_from_smiles(text) for text in examples.values()]
query_highlights = [sorted({atom for match in mol.GetSubstructMatches(alcohol_query)
                            for atom in match}) for mol in query_molecules]
query_legends = [f"{label}: {'match' if hits[label] else 'no match'}" for label in examples]
query_drawing = rdMolDraw2D.MolDraw2DCairo(900, 440, 300, 220)
query_drawing.DrawMolecules(query_molecules, legends=query_legends,
                            highlightAtoms=query_highlights)
query_drawing.FinishDrawing()
display(Image(data=query_drawing.GetDrawingText()))
assert [len(mol.GetSubstructMatches(alcohol_query)) for mol in query_molecules] == [1, 0, 0, 0, 2, 0]

**Guided exercise:** include phenols by adding a second query `c[OX2H1]`. Predict which additional record will match, then verify. **Selected answer:** phenol is added; the ether still lacks the required O–H group. If the project concerns reactivity, follow up with the full chemical environment, conditions, and experimental evidence.

**Reaction records**

Reaction SMILES has the form `reactants>agents>products`; dots separate components. A reaction record describes the written participants. It does not predict reaction conditions, mechanism, feasibility, yield, or stereoselectivity.

The S_N2 example below records methyl chloride plus methoxide giving dimethyl ether and chloride. `useSmiles=True` tells the RDKit reaction parser to interpret the components as molecule SMILES, so this is a **reaction SMILES record**, not a general SMARTS transformation rule.

In [ ]:
sn2_reaction_smiles = "CCl.C[O-]>>COC.[Cl-]"
sn2_reaction = rdChemReactions.ReactionFromSmarts(sn2_reaction_smiles, useSmiles=True)
Draw.ReactionToImage(sn2_reaction, subImgSize=(220, 150))

**Agents and a second reaction record**

This Fischer esterification record includes an acid catalyst in the agent field. The reaction is reversible in practice, and its conditions and equilibrium are not encoded by the arrow. Use hydronium as a simple representation of an acid catalyst; this does not specify a particular solvent, acid concentration, or catalytic cycle.

In [ ]:
esterification_smiles = "CC(=O)O.CC(C)CCO>[OH3+]>CC(=O)OCCC(C)C.O"
esterification_reaction = rdChemReactions.ReactionFromSmarts(esterification_smiles, useSmiles=True)
print("Reactant templates:", esterification_reaction.GetNumReactantTemplates())
print("Agent templates:", esterification_reaction.GetNumAgentTemplates())
Draw.ReactionToImage(esterification_reaction, subImgSize=(240, 170))

**Optional: a mapped transformation template**

RDKit reaction SMARTS uses atom-map labels (`:1`, `:2`, etc.) to relate reactant and product atoms. These labels are identifiers, not charges or atom indices. The following narrowly scoped template changes a bond and explicitly neutralizes oxygen; leaving its charge unspecified can preserve the reactant charge.

`RunReactants` applies the written rule. Its products need sanitization and chemical validation; matching a template is not evidence that a reaction succeeds in the laboratory. [RDKit reaction SMARTS](https://www.rdkit.org/docs/RDKit_Book.html#reaction-smarts).

In [ ]:
sn2_template = rdChemReactions.ReactionFromSmarts(
    "[CH3:1][Cl:2].[CH3:3][O-:4]>>[CH3:1][O+0:4][CH3:3].[Cl-:2]"
)
product_sets = sn2_template.RunReactants((molecule_from_smiles("CCl"),
                                         molecule_from_smiles("C[O-]")))
assert len(product_sets) == 1
product_smiles = []
for product in product_sets[0]:
    Chem.SanitizeMol(product)
    product_smiles.append(Chem.MolToSmiles(product))
assert sorted(product_smiles) == ["COC", "[Cl-]"]
print("Template products:", product_smiles)

## 1.6. Chemical Structure File Formats

### 1.6.1. XYZ Files

#### 1.6.1.1. Introduction

A basic XYZ record contains atom identities and Cartesian coordinates, conventionally in angstroms. It does **not** explicitly store bonds, bond orders, formal charges, or a unit declaration. Extended XYZ variants add fields, but software must agree on their meaning. Do not treat an XYZ conversion as a lossless way to store a chemical graph.

#### 1.6.1.2. Structure of an XYZ File

A basic single-frame XYZ file has:

1. An integer atom count.
2. A comment line (which may be empty).
3. One `element x y z` row per atom.

Hydrogens must appear as rows when their coordinates are required. Confirm the producer's units before interpreting distances.

Example: .xyz file of water molecule:

```text
3
Water; coordinates in angstrom
O    0.000000    0.000000    0.000000
H    0.758602    0.601435    0.000000
H   -0.758602    0.601435    0.000000
```

These rows describe an illustrative geometry for one oxygen and two hydrogen atoms. All z coordinates are zero: a planar molecule can still have a meaningful three-dimensional coordinate record. The coordinates alone do not specify which atoms are bonded.

#### 1.6.1.3. Reading and Writing XYZ Files

**Reading XYZ Files with RDKit**

`Chem.MolFromXYZBlock` reads atoms and coordinates but does not assign bonds. We then use `rdDetermineBonds.DetermineBonds` to infer connectivity and bond orders, supplying the known total charge.

Bond inference is a model, not an identity check. Incomplete hydrogen coordinates, unusual bonding, metals, incorrect charge, or distorted geometries can produce failure or wrong assignments. A single distance cutoff cannot assign general chemical bond orders. Prefer a trusted connection table when one is available. [RDKit bond inference](https://www.rdkit.org/docs/source/rdkit.Chem.rdDetermineBonds.html).

In [ ]:
xyz_string = """3
Water; coordinates in angstrom
O    0.000000    0.000000    0.000000
H    0.758602    0.601435    0.000000
H   -0.758602    0.601435    0.000000
"""

In [ ]:
water_coordinates_only = Chem.MolFromXYZBlock(xyz_string)
if water_coordinates_only is None:
    raise ValueError("Invalid XYZ record.")
print("Atoms:", water_coordinates_only.GetNumAtoms())
print("Bonds before inference:", water_coordinates_only.GetNumBonds())

water_mol = Chem.Mol(water_coordinates_only)
rdDetermineBonds.DetermineBonds(water_mol, charge=0)
Chem.SanitizeMol(water_mol)
assert water_mol.GetNumBonds() == 2
assert rdMolDescriptors.CalcMolFormula(water_mol) == "H2O"
print("Bonds after inference:", water_mol.GetNumBonds())

In [ ]:
# Make a separate 2D depiction while preserving the XYZ conformer.
water_2d = Chem.Mol(water_mol)
rdDepictor.Compute2DCoords(water_2d)
Draw.MolToImage(water_2d, size=(200, 160))

**Optional programming extension: coordinates with known connectivity**

If a trusted SMILES is available, attach coordinates using a **known atom mapping**. SMILES order and XYZ row order need not agree, even for two carbon atoms in the same molecule. Matching atom counts and element symbols is necessary but insufficient to establish the mapping.

The helper below takes `xyz_to_mol`, where entry `i` names the atom in the hydrogen-expanded RDKit graph corresponding to XYZ row `i`. It checks row count, numerical coordinates, the mapping, and element identities. It cannot prove that same-element atoms were mapped correctly or that coordinates agree with supplied stereochemistry; those require provenance or further chemical checks.

For our water record, XYZ rows are O, H, H and `Chem.AddHs(molecule_from_smiles("O"))` has the same atom order. Therefore the known mapping is `[0, 1, 2]`. No embedding is needed when coordinates already exist.

In [ ]:
water_template = Chem.AddHs(molecule_from_smiles("O"))
print([(atom.GetIdx(), atom.GetSymbol()) for atom in water_template.GetAtoms()])

In [ ]:
def mol_from_xyz_with_smiles(xyz_text, smiles, xyz_to_mol):
    """Attach one basic XYZ frame using an externally established atom mapping."""
    lines = xyz_text.strip().splitlines()
    if len(lines) < 3:
        raise ValueError("XYZ requires an atom count, comment, and atom rows.")
    try:
        atom_count = int(lines[0])
    except ValueError as error:
        raise ValueError("XYZ atom count must be an integer.") from error
    rows = lines[2:]
    if atom_count <= 0 or len(rows) != atom_count:
        raise ValueError("XYZ atom count does not match the number of rows.")
    mol = Chem.AddHs(molecule_from_smiles(smiles))
    if mol.GetNumAtoms() != atom_count:
        raise ValueError("XYZ and hydrogen-expanded SMILES have different atom counts.")
    if (len(xyz_to_mol) != atom_count
            or any(type(index) is not int for index in xyz_to_mol)
            or sorted(xyz_to_mol) != list(range(atom_count))):
        raise ValueError("xyz_to_mol must be a permutation of all atom indices.")
    conformer = Chem.Conformer(atom_count)
    conformer.Set3D(True)
    for row_index, row in enumerate(rows):
        fields = row.split()
        if len(fields) != 4:
            raise ValueError("Each XYZ atom row must contain element, x, y, z.")
        atom_index = xyz_to_mol[row_index]
        if fields[0] != mol.GetAtomWithIdx(atom_index).GetSymbol():
            raise ValueError(f"Element mismatch at XYZ row {row_index}.")
        position = np.array([float(value) for value in fields[1:]], dtype=float)
        if not np.all(np.isfinite(position)):
            raise ValueError("Coordinates must be finite numbers.")
        conformer.SetAtomPosition(atom_index, position.tolist())
    mol.RemoveAllConformers()
    mol.AddConformer(conformer, assignId=True)
    return mol

In [ ]:
water_from_template = mol_from_xyz_with_smiles(xyz_string, "O", [0, 1, 2])
np.testing.assert_allclose(water_from_template.GetConformer().GetPositions(),
                           water_coordinates_only.GetConformer().GetPositions())
assert water_from_template.GetNumBonds() == 2

# A wrong mapping is reported rather than silently assigning O coordinates to H.
try:
    mol_from_xyz_with_smiles(xyz_string, "O", [1, 0, 2])
except ValueError as error:
    print("Expected mapping error:", error)

**Writing XYZ Files with RDKit**

`Chem.MolToXYZBlock` exports the current coordinates to text; `Chem.MolToXYZFile(mol, "my_structure.xyz")` writes a file. A conformer must exist first. The XYZ output loses the graph's connectivity, charges, and stereo annotations. Retain the originating MOL/SDF and metadata when those are needed. [RDKit file I/O](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html).

In [ ]:
# Export the known water coordinates; do not generate unrelated random coordinates.
exported_xyz = Chem.MolToXYZBlock(water_from_template)
print(exported_xyz)
reread_coordinates = Chem.MolFromXYZBlock(exported_xyz)
assert reread_coordinates is not None
assert reread_coordinates.GetNumBonds() == 0
np.testing.assert_allclose(reread_coordinates.GetConformer().GetPositions(),
                           water_from_template.GetConformer().GetPositions(), atol=1e-6)

### 1.6.2. Other Chemical Structure File Formats

Choose a format according to the information you need to preserve:

| Format | Main contents and limitations |
|---|---|
| SMILES | Molecular graph with optional isotope/stereo information; ordinary SMILES has no coordinates |
| XYZ | Elements and Cartesian coordinates; no explicit bonding in the basic format |
| MOL | One connection table with bonds, annotations, and 2D or 3D coordinates; V2000 and V3000 variants |
| SDF | One or more MOL records plus named data fields; useful for compound collections |
| PDB / PDBx/mmCIF | Biomolecular coordinates and metadata; PDB is the legacy format and PDBx/mmCIF is the wwPDB standard |
| CIF | Crystallographic data, such as unit cell, symmetry, and atom sites; positions may be fractional coordinates |

MOL is not restricted to 2D. A PDB file does not necessarily provide a complete small-molecule bond-order model. Cartesian coordinates, fractional coordinates, and a chemical connection table answer different questions.

Sources: [RDKit molecular file I/O](https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html), [wwPDB file formats](https://www.wwpdb.org/documentation/file-format), and [IUCr CIF standard](https://www.iucr.org/what-we-do/digital-standards/cif).

In [ ]:
# A MOL round-trip retains this graph and its 3D conformer.
ethanol_molblock = Chem.MolToMolBlock(ethanol_3d)
ethanol_round_trip = Chem.MolFromMolBlock(ethanol_molblock, removeHs=False)
assert ethanol_round_trip is not None
assert ethanol_round_trip.GetNumBonds() == ethanol_3d.GetNumBonds()
assert ethanol_round_trip.GetConformer().Is3D()
assert Chem.MolToSmiles(ethanol_round_trip) == Chem.MolToSmiles(ethanol_3d)
np.testing.assert_allclose(ethanol_round_trip.GetConformer().GetPositions(),
                           ethanol_3d.GetConformer().GetPositions(), atol=1e-4)
print("MOL round-trip preserved the ethanol graph and 3D coordinates within file precision.")

### Practice and self-check

1. Parse and draw `CC(C)CC` and `CC(C)C(C)C`. Check their formulas and explain why the latter cannot be called 2-methylbutane.
2. Compare `n1ccccc1` (pyridine) with `[nH]1cccc1` (pyrrole). Count ring atoms and explain the bracket hydrogen.
3. Explain why `@` can correspond to R in one SMILES and S in another. Verify one reordered alanine representation using `cip_labels`.
4. Canonicalize `CC(=O)O` and `CC(=O)[O-]`. Should these become the same string? What information differs?
5. Test the alcohol query against phenol (`Oc1ccccc1`). Predict the result from the `CX4` requirement, then decide whether a phenol query needs different constraints.
6. Why does reading an XYZ record return zero bonds? Why is matching only the element sequence not enough to attach arbitrary XYZ coordinates to a SMILES graph?
7. Paclitaxel's formula matches the expected formula. Is that alone enough to establish its identity? Name another check used here.
8. Explain what is missing from a reaction record before it can support an experimental claim.

<details><summary>Selected answers</summary>

1. C5H12 versus C6H14; the second graph is 2,3-dimethylbutane.
2. Pyridine has six ring atoms; pyrrole has five. `[nH]` specifies pyrrole's N-H aromatic nitrogen.
3. `@` and `@@` refer to SMILES neighbor order; R/S refers to CIP priorities.
4. No. The structures differ in protonation and formal charge.
5. False: an aromatic carbon does not match `CX4`; a phenol query can use `c[OX2H1]`.
6. Basic XYZ omits bonds. Two atoms of the same element may have different environments, so their coordinates still require the correct mapping.
7. No; the formula does not distinguish isomers. This notebook also checks the stereochemistry-sensitive InChIKey against the PubChem record.
8. Conditions, experimental evidence, and the scope of any claimed yield or selectivity, among other metadata.

</details>

**Takeaway:** validate chemical identity, representation, and coordinates separately. Preserve provenance, units, atom mappings, and toolkit versions when exchanging molecular data. Continue with [Chapter 2](Chapter02_Part1.ipynb).
